In [19]:
#import tensorflow as tf 
#from tensorflow.keras import layers, models, datasets 
import numpy as np 
import matplotlib.pyplot as plt 
# Load the IMDB reviews dataset, which comes with an index of words 
num_words = 10000  # Keep the top 10,000 most frequent words 
(train_data, train_labels), (test_data, test_labels) = datasets.imdb.load_data(num_words=num_words) 
# The data is already pre-processed as sequences of integer word indices 
print("Training data shape:", train_data.shape) 
print("First review as integers:", train_data[0]) 
# Get the word index dictionary to decode reviews 
word_index = datasets.imdb.get_word_index() 
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()]) 
# Decode the first review. Note: indices are offset by 3 (0,1,2 are reserved) 
decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in train_data[0]]) 
print("\nDecoded review:") 
print(decoded_review)

# Pad sequences to make them all the same length (e.g., 500 words) 
maxlen = 500 
train_data = tf.keras.preprocessing.sequence.pad_sequences(train_data, maxlen=maxlen, 
padding='post', truncating='post') 
test_data = tf.keras.preprocessing.sequence.pad_sequences(test_data, maxlen=maxlen, 
padding='post', truncating='post') 
print("Padded training data shape:", train_data.shape)

# Hyperparameters 
embedding_dim = 100  # Size of the embedding vector for each word 
vocab_size = num_words + 1 # +1 for the index 0 which is used for padding 
 
model = models.Sequential() 
# Add an Embedding layer. This is the key step. 
# It will create a lookup table of size (vocab_size, embedding_dim) 
model.add(layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, 
input_length=maxlen)) 
 
# After embedding, the output is a 2D tensor of shape (500, 100). We need to flatten it. 
# We can use a GlobalAveragePooling1D to average the 500 embedding vectors into a single 
#100-dim vector. 
# This is efficient and often works well. 
model.add(layers.GlobalAveragePooling1D()) 
 
# Add a Dense layer for classification 
model.add(layers.Dense(16, activation='relu')) 
# Output layer: 1 neuron for binary classification, with sigmoid activation 
model.add(layers.Dense(1, activation='sigmoid')) 
 
model.summary() 
 
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

history = model.fit(train_data, train_labels, 
                    epochs=10, 
                    batch_size=32, 
                    validation_split=0.2) # Use 20% of training data for validation 

# Plot history 
plt.figure(figsize=(12, 4)) 
plt.subplot(1, 2, 1) 
plt.plot(history.history['accuracy'], label='Training Acc') 
plt.plot(history.history['val_accuracy'], label='Validation Acc') 
plt.title('Training and Validation Accuracy') 
plt.legend() 
plt.subplot(1, 2, 2) 
plt.plot(history.history['loss'], label='Training Loss') 
plt.plot(history.history['val_loss'], label='Validation Loss') 
plt.title('Training and Validation Loss') 
plt.legend() 
plt.show() 
# Evaluate on test set 
test_loss, test_acc = model.evaluate(test_data, test_labels) 
print(f'\nTest Accuracy: {test_acc:.4f}')

# Get the weights from the Embedding layer 
embedding_layer = model.layers[0] 
embeddings = embedding_layer.get_weights()[0] 
print("Embeddings shape:", embeddings.shape) # (10001, 100) 
# Now you could use a tool like Projector (https://projector.tensorflow.org/) to visualize these 
#100-dimensional vectors in 2D or 3D space. 
# Words with similar meaning should be closer together in this space.

NameError: name 'datasets' is not defined